**# BorrowBox V2.1 — Database / Entity Schema**

**\*\*Status:\*\*** Planning baseline  

**\*\*Scope:\*\*** V2.1 Community + Ownership Foundation  

**\*\*Important:\*\*** This is the authoritative V2.1 domain/data-model design. V2.1 starts from a fresh database/schema; V1 data migration is out of scope.

**---**

**## 1. V2.1 goal**

Make BorrowBox genuinely community-based while preserving a single source of truth for physical assets owned by a user.

The key domain distinction is:

\`\`\`text

Asset = what the user owns

Asset Unit = an individually trackable physical unit

Community Listing = where the owner makes that asset available

Transaction = one particular borrowing event

\`\`\`

A single owned asset pool can therefore be listed in multiple communities without creating duplicate inventory.

Example:

\`\`\`text

Ahmed owns:

    Football × 2

Listed in:

    Hostel A

    Hostel B

    Office

This is ONE owned asset pool.

\`\`\`

**---**

**# 2. Entity overview**

V2.1 introduces or formalizes these core entities:

\`\`\`text

User

  │

  ├── Membership ───────► Community

  │                          │

  │                          ├── CommunityRule

  │                          └── CommunityListing

  │

  └── Asset

        │

        ├── AssetUnit

        │

        └── CommunityListing

\`\`\`

Later V2 milestones attach transactions to AssetUnit/Asset and then add:

\`\`\`text

Transaction

Message

Handover

Evidence

Notification

ReputationEvent

Flag

\`\`\`

Those later entities are deliberately outside the V2.1 database implementation scope unless needed as compatibility placeholders.

**---**

**# 3. User**

**## Purpose**

Represents the global BorrowBox account/person.

**### Proposed fields**

\| Field | Type concept | Null? | Notes |

\|---|---|---:|---|

\| id | BIGINT | NO | Primary key |

\| full\_name | VARCHAR | NO | Display name |

\| email | VARCHAR | NO | Unique login identity |

\| password\_hash | VARCHAR | NO | Authentication credential |

\| status | ENUM/string | NO | ACTIVE / SUSPENDED / etc. |

\| created\_at | TIMESTAMP | NO | Creation time |

\| updated\_at | TIMESTAMP | NO | Last update |

**### Important rule**

Do **\*\*not\*\*** store community-specific roles such as:

\`\`\`text

student

employee

resident

hostel room

department

\`\`\`

directly on User.

Those belong to Membership.

**---**

**# 4. Community**

**## Purpose**

Represents a bounded real-world/social group.

Examples:

\`\`\`text

Hostel Block B

CSE Department

Engineering Office

Tower B Residents

Weekend Football Club

\`\`\`

**### Proposed fields**

\| Field | Type concept | Null? | Notes |

\|---|---|---:|---|

\| id | BIGINT | NO | Primary key |

\| name | VARCHAR | NO | Community name |

\| description | TEXT | YES | Community description |

\| type | ENUM/string | NO | HOSTEL / OFFICE / COLLEGE / HOUSING / CLUB / FRIENDS / OTHER |

\| status | ENUM/string | NO | ACTIVE / ARCHIVED |

\| created\_by | BIGINT FK User | NO | Creator/initial manager |

\| location\_latitude | DECIMAL | YES | Future location verification |

\| location\_longitude | DECIMAL | YES | Future location verification |

\| location\_radius\_m | INTEGER | YES | Future geofence/radius |

\| created\_at | TIMESTAMP | NO | Creation time |

\| updated\_at | TIMESTAMP | NO | Last update |

**### Rule**

Community location is for membership/community verification where appropriate. It must not imply continuous member tracking.

**---**

**# 5. Membership**

**## Purpose**

The first-class relationship between a User and a Community.

\`\`\`text

User 1 ────< Membership >──── 1 Community

\`\`\`

**### Proposed fields**

\| Field | Type concept | Null? | Notes |

\|---|---|---:|---|

\| id | BIGINT | NO | Primary key |

\| user\_id | BIGINT FK User | NO | Member |

\| community\_id | BIGINT FK Community | NO | Community |

\| role | ENUM/string | NO | MEMBER / MANAGER initially |

\| status | ENUM/string | NO | PENDING / ACTIVE / SUSPENDED / LEFT / REJECTED |

\| verification\_method | ENUM/string | YES | INVITE / LOCATION / MANAGER\_APPROVAL / ADMIN |

\| verified\_at | TIMESTAMP | YES | Verification time |

\| verified\_by | BIGINT FK User | YES | Manager/admin if applicable |

\| joined\_at | TIMESTAMP | YES | Activation/join time |

\| created\_at | TIMESTAMP | NO | Record creation |

\| updated\_at | TIMESTAMP | NO | Last update |

**### Unique constraint**

\`\`\`text

UNIQUE(user\_id, community\_id)

\`\`\`

A user should have at most one active membership record per community.

**### Important design principle**

Membership is where community-specific identity lives.

Example:

\`\`\`text

Ahmed

 ├── Hostel B membership

 │     role = MEMBER

 │     context = Resident / Floor 3 / Room B-302

 │

 └── CSE membership

       role = MEMBER

       context = Student / Year 4 / Section A

\`\`\`

**---**

**# 6. Membership attributes**

Different community types need different contextual information.

**### Hostel**

\`\`\`text

block

floor

room

college

year

\`\`\`

**### Office**

\`\`\`text

department

team

designation

\`\`\`

**### College**

\`\`\`text

program

year

section

\`\`\`

**### Housing**

\`\`\`text

tower

floor

flat

resident\_type

\`\`\`

**## LOCKED V2.1 representation**

Use a MySQL \`JSON\` column on \`Membership\`:

\`\`\`text

context\_metadata JSON

\`\`\`

Example:

\`\`\`json

{

  "block": "B",

  "floor": 3,

  "room": "B-302"

}

\`\`\`

This keeps community-specific context flexible without requiring many nullable columns or schema changes for every community type.

The exact allowed keys and validation rules are application-level policy and may evolve later.

## LOCKED JPA JSON mapping rule

`Membership.context_metadata` is stored in MySQL using the native `JSON` column type.

For the Java/JPA implementation, prefer Hibernate's native JSON mapping:

```java
@JdbcTypeCode(SqlTypes.JSON)
private Map<String, Object> contextMetadata;
```

Do not introduce Hypersistence Utils or another JSON-mapping dependency unless native Hibernate mapping is demonstrated to be insufficient in this repository.

Required integration verification:

```text
1. Persist a Membership with non-empty context_metadata.
2. Read the Membership back through JPA.
3. Verify the JSON structure and values are preserved.
4. Run the MySQL-backed integration test successfully.
```


**# 7. CommunityRule**

## V2.1 scope boundary

`CommunityRule` is part of the broader V2.1 roadmap but is **not implemented in V2.1.1**.

The first rules slice will be introduced after the Community + Membership foundation is stable.

Do not create a rule engine, rule table, or rule-management endpoints inside V2.1.1.

The Community entity may store the structural fields needed for later policy enforcement, but policy behavior belongs to the dedicated later slice.

**---**

**# 8. Asset**

**## Purpose**

Represents what the user actually owns.

**### Proposed fields**

\| Field | Type concept | Null? | Notes |

\|---|---|---:|---|

\| id | BIGINT | NO | Primary key |

\| owner\_id | BIGINT FK User | NO | Owner |

\| title | VARCHAR | NO | User-facing name |

\| description | TEXT | YES | Description |

\| category\_id | BIGINT FK Category | YES | Reuse current category model if possible |

\| status | ENUM/string | NO | ACTIVE / ARCHIVED |

\| created\_at | TIMESTAMP | NO | Creation time |

\| updated\_at | TIMESTAMP | NO | Last update |

**## LOCKED quantity rule**

\`Asset\` does **\*\*not\*\*** contain a \`total\_quantity\` column.

Physical quantity is represented entirely by \`AssetUnit\` rows.

\`\`\`text

Total units = COUNT(non-ARCHIVED AssetUnits)

Available   = COUNT(AVAILABLE AssetUnits)

Borrowed    = COUNT(BORROWED AssetUnits)

\`\`\`

The Asset is not community-specific.

Example:

\`\`\`text

Ahmed owns:

Football × 2

\`\`\`

This is one Asset regardless of where it is listed.

**# 9. AssetUnit**

**## Purpose**

Represents one physically trackable unit within an Asset.

Example:

\`\`\`text

Asset:

Football × 2

Units:

  Unit 1

  Unit 2

\`\`\`

**### Proposed fields**

\| Field | Type concept | Null? | Notes |

\|---|---|---:|---|

\| id | BIGINT | NO | Primary key |

\| asset\_id | BIGINT FK Asset | NO | Parent asset |

\| unit\_identifier | VARCHAR | YES | Optional human/business identifier |

\| status | ENUM/string | NO | AVAILABLE / BORROWED / NOT\_AVAILABLE / LOST / DAMAGED / ARCHIVED |

\| condition | VARCHAR/JSON | YES | Future condition tracking |

\| created\_at | TIMESTAMP | NO | Creation time |

\| updated\_at | TIMESTAMP | NO | Last update |

**## LOCKED materialization rule**

When an Asset is created with quantity \`N\`, the backend must perform the following as **\*\*one atomic database transaction\*\***:

\`\`\`text

BEGIN TRANSACTION

    ↓

insert 1 Asset row

    ↓

insert exactly N AssetUnit rows

    ↓

COMMIT

\`\`\`

AssetUnit materialization is mandatory and immediate; no alternative creation model is permitted.

The UI may present the units as one grouped asset while the backend tracks each physical unit individually.

**# 10. CommunityListing**

**## Purpose**

Connects an owned Asset to a Community.

\`\`\`text

Asset 1 ────< CommunityListing >──── 1 Community

\`\`\`

**### Proposed fields**

\| Field | Type concept | Null? | Notes |

\|---|---|---:|---|

\| id | BIGINT | NO | Primary key |

\| asset\_id | BIGINT FK Asset | NO | Owned asset |

\| community\_id | BIGINT FK Community | NO | Visibility context |

\| listing\_status | ENUM/string | NO | LISTED / UNLISTED |

\| listed\_at | TIMESTAMP | NO | First listing time |

\| updated\_at | TIMESTAMP | NO | Last update |

**### Unique constraint**

\`\`\`text

UNIQUE(asset\_id, community\_id)

\`\`\`

An Asset should not have duplicate listings inside the same Community.

**## LOCKED owner-membership authorization rule**

A \`CommunityListing\` may be created or activated only when:

\`\`\`text

Asset.owner\_id

    ↓

ACTIVE Membership

    ↓

Membership.community\_id = CommunityListing.community\_id

\`\`\`

The backend service/transaction layer must verify this before inserting or activating the listing.

The database separately enforces the ordinary foreign keys:

\`\`\`text

CommunityListing.asset\_id → Asset.id

CommunityListing.community\_id → Community.id

\`\`\`

This is a cross-table business authorization rule, not a simple database CHECK constraint.

**## LOCKED public API delivery rule**

For V2.1, public Explore/search responses must aggregate matching Assets into **\*\*one summary per CommunityListing\*\***.

Individual \`AssetUnit\` rows and unit IDs are internal implementation details and must not be exposed in public catalog/search responses.

A public summary may contain:

\`\`\`text

asset id

title

description

category

listing status

total units

available units

borrowed units

\`\`\`

The exact DTO shape may evolve, but the aggregation rule is fixed.

**### Important rule**

A CommunityListing has no independent quantity.

Availability comes from the shared AssetUnit pool.

**# 11. Availability model**

This is a central rule.

Suppose:

\`\`\`text

Asset:

Football × 2

Listings:

Community A

Community B

Community C

\`\`\`

Initially:

\`\`\`text

Total units = 2

Borrowed = 0

Available = 2

\`\`\`

After one unit is borrowed:

\`\`\`text

Total = 2

Borrowed = 1

Available = 1

\`\`\`

All three community listings must reflect:

\`\`\`text

1 available

\`\`\`

After both units are borrowed:

\`\`\`text

Available = 0

\`\`\`

All three listings reflect:

\`\`\`text

Not available / fully borrowed

\`\`\`

When one returns:

\`\`\`text

Available = 1

\`\`\`

All listings update automatically.

**### Authority**

The database/backend is authoritative.

The frontend must never independently decide availability.

**---**

**# 12. Asset status vs availability**

Do not collapse every concept into one status.

**### Asset lifecycle**

\`\`\`text

ACTIVE

ARCHIVED

\`\`\`

**### Community listing state**

\`\`\`text

LISTED

UNLISTED

\`\`\`

**### Physical unit state**

\`\`\`text

AVAILABLE

BORROWED

NOT\_AVAILABLE

LOST

DAMAGED

ARCHIVED

\`\`\`

**### Aggregate availability**

Availability is derived from AssetUnit rows:

\`\`\`text

Total units = COUNT(non-ARCHIVED AssetUnits)

Available   = COUNT(AVAILABLE AssetUnits)

Borrowed    = COUNT(BORROWED AssetUnits)

\`\`\`

The frontend does not maintain an independent availability truth.

The backend/database is authoritative.

**# 13. Relationships summary**

\`\`\`text

USER

 ├──< MEMBERSHIP >── COMMUNITY

 │                      │

 │                      ├──< COMMUNITY\_RULE

 │                      └──< COMMUNITY\_LISTING

 │

 └──< ASSET

        │

        ├──< ASSET\_UNIT

        │

        └──< COMMUNITY\_LISTING >── COMMUNITY

\`\`\`

Cardinality:

\`\`\`text

User 1 → many Memberships

Community 1 → many Memberships

User 1 → many Assets

Asset 1 → many AssetUnits

Asset 1 → many CommunityListings

Community 1 → many CommunityListings

Community 1 → many CommunityRules

\`\`\`

**---**

**# 14. Foreign keys and deletion behavior**

Recommended baseline:

**### User → Membership**

Do not hard-delete memberships casually.

Prefer status changes:

\`\`\`text

ACTIVE → LEFT

ACTIVE → SUSPENDED

\`\`\`

**### Community → Membership**

Preserve membership history where possible.

**### User → Asset**

An Asset belongs to its owner.

Do not cascade-delete assets merely because a user leaves one community.

**### Asset → CommunityListing**

Leaving a community should remove/unlist the membership/listing relationship, not delete the underlying asset.

**### Asset → AssetUnit**

Deleting an Asset should only be permitted where no historical transactions depend on its units. Later, archival is preferable.

**### Community → CommunityRule**

Rules may be updated/deactivated rather than destructively deleted if historical auditability matters.

**---**

**# 15. Indexes**

Likely high-value indexes:

\`\`\`text

memberships:

  UNIQUE(user\_id, community\_id)

  INDEX(community\_id, status)

  INDEX(user\_id, status)

assets:

  INDEX(owner\_id, status)

asset\_units:

  INDEX(asset\_id, status)

community\_listings:

  UNIQUE(asset\_id, community\_id)

  INDEX(community\_id, listing\_status)

  INDEX(asset\_id, listing\_status)

community\_rules:

  INDEX(community\_id, rule\_type)

\`\`\`

These should be validated against actual query patterns before being implemented.

**---**

**# 16. Concurrency / race condition preparation**

The schema must eventually support safe scarce-resource reservation.

Scenario:

\`\`\`text

Football × 1 available

12:01:04.123 → Salah requests

12:01:04.124 → Ahmed requests

\`\`\`

Client timestamps are not authoritative.

The backend/database must:

1\. Establish server-side ordering.

2\. Lock/check the relevant AssetUnit or inventory row.

3\. Verify availability.

4\. Create the reservation/transaction for the first valid request.

5\. Ensure another concurrent request cannot reserve the same unit.

This belongs primarily to V2.2, but V2.1's data model must not prevent it.

**---**

**# 17. V2.1 scope boundary**

**## Build now**

\`\`\`text

User

Community

Membership

Community roles

Community types

Community rules foundation

Asset

AssetUnit

CommunityListing

Community-scoped Explore

Community-scoped inventory

Membership management basics

\`\`\`

**## Defer**

\`\`\`text

Transaction

Messages

Handover

Evidence/photos

Notifications

Reputation

Flags

AI inspection

Tribunal

\`\`\`

The transaction model should be designed now but implemented after the community/ownership boundary is stable.

**---**

**# 18. V2.1 acceptance criteria**

V2.1 is complete when all of the following are true:

\`\`\`text

1\. A user can create a community.

2\. The creator becomes its manager.

3\. A user can join/leave according to community policy.

4\. Membership has a community-specific role/context.

5\. A community can define basic rules.

6\. A user can create an owned asset.

7\. One asset can be listed in multiple selected communities.

8\. The asset is not duplicated per community.

9\. Availability is shared across all listings.

10\. Borrowing one physical unit reduces availability everywhere.

11\. Returning it restores availability everywhere.

12\. Community Explore only exposes listings for the active community.

13\. Users cannot view community inventory without appropriate membership/authorization.

14\. Concurrent availability decisions are owned by the backend/database.

15\. The entire feature works through UI → API → database with tests and Docker verification.

\`\`\`

**---**

**# 19. V2 database initialization — LOCKED**

V2.1 starts from a **\*\*fresh database/schema\*\***.

There is no V1-to-V2 migration layer in the V2 application baseline.

The V1 development/demo database remains preserved in Git/history, but V2 does not import its rows.

**## Fresh initialization flow**

\`\`\`text

fresh database

   ↓

create V2.1 schema

   ↓

seed deterministic users

   ↓

seed deterministic communities

   ↓

seed memberships/roles

   ↓

seed assets + AssetUnits

   ↓

seed CommunityListings

   ↓

start V2.1

\`\`\`

**## Rule**

Do not add:

\`\`\`text

V1 migration scripts

V1 compatibility queries

legacy Group → Community conversion

legacy Item → Asset conversion

legacy BorrowRequest → Transaction conversion

\`\`\`

unless a future product decision explicitly reintroduces data import.

V1 is a historical release, not a V2 runtime dependency.

**# 20. Locked design decisions for V2.1

## LOCKED

### A. Membership context

Use:

```text
Membership.context_metadata JSON
```

using MySQL's native JSON type and Hibernate native JSON mapping.

### B. AssetUnit materialization

Create exactly `N` AssetUnit rows immediately when an Asset with quantity `N` is created, in the same atomic database transaction.

### C. CommunityListing owner authorization

A listing can only be created or activated when the Asset owner has an ACTIVE Membership in the target Community. Enforce this server-side in the backend service/use-case transaction.

### D. Asset quantity source of truth

Do not store `total_quantity` on Asset. Quantity is derived from AssetUnits.

### E. Public V2.1 inventory API

Public Explore/search responses aggregate AssetUnits into a single Asset summary per CommunityListing and never expose AssetUnit IDs.

### F. V2 database initialization

V2.1 starts from a fresh database and deterministic development seed data. V1 rows are not imported and V1 migration is not a runtime concern.

### G. Community-name uniqueness

The same creator/manager cannot have two ACTIVE Communities with the same normalized name.

Different creators may use the same name.

MySQL enforcement is locked to a database-level normalized active-name key:

```text
active_name_key =
    LOWER(TRIM(name)) when status = ACTIVE
    NULL otherwise

UNIQUE(created_by, active_name_key)
```

This allows multiple archived communities with the same name while preventing duplicate ACTIVE communities for one creator.

### H. Community admission

BorrowBox has **no OPEN_FOR_ALL community mode**.

V2.1 defines two admission modes:

```text
MANAGER_APPROVAL
LOCATION_VERIFIED
```

`MANAGER_APPROVAL` is the default.

### I. Manager-approval join flow

For `MANAGER_APPROVAL`:

```text
join request
→ Membership.status = PENDING
→ manager reviews
→ ACTIVE or REJECTED
```

A manager decision must set `verified_by` to the approving manager.

### J. Location-verified join flow

For `LOCATION_VERIFIED`:

```text
user explicitly grants location for this join attempt
→ server checks distance from community location
→ inside configured radius → ACTIVE
→ outside radius → PENDING for manager review
```

Location is checked for the operation only. BorrowBox must not continuously track members.

The V2.1.2 verification mechanism is a latitude/longitude + radius check. QR/NFC and other stronger verification mechanisms remain future enhancements.

### K. Verified-by authorization

`Membership.verified_by` must reference a User who has an ACTIVE `MANAGER` Membership in the same Community.

This is a cross-table business rule and must be enforced server-side.

### L. CommunityRule scope

`CommunityRule` is **not part of V2.1.1**. It will be implemented in a later V2.1 rules slice.

### M. AssetUnit public visibility

Public catalog/search always returns aggregate asset information.

Individual AssetUnit identity is internal. Owner/manager unit-level views may be introduced later when advanced condition tracking is built.

### N. Seed mechanism

Use a Spring `ApplicationRunner`/equivalent initializer gated by:

```text
borrowbox.seed.enabled=true
```

Seed identities must be deterministic and the initializer must be idempotent.

### O. Schema initialization

Use a version-controlled V2 schema definition (`schema.sql`) with:

```text
spring.jpa.hibernate.ddl-auto=validate
```

for the V2.1 application baseline.

Do not introduce Flyway/Liquibase solely for V2.1 migration because V2 starts from a fresh database.

### P. Soft deletion

V2.1 uses lifecycle/status fields (`ACTIVE/ARCHIVED`, `LISTED/UNLISTED`, unit status) as the authoritative lifecycle mechanism.

Do not add redundant `deleted_at` columns unless a later audit requirement makes them necessary.

**# 21. Locked architectural invariants**

These are hard design rules unless explicitly changed in the Decisions log.

\`\`\`text

1\. A User is a person/member, not permanently a borrower or lender.

2\. Community membership is first-class.

3\. Community-specific role/context belongs to Membership.

4\. Membership context is stored in MySQL JSON as context\_metadata.

5\. Asset ownership is independent of Community.

6\. CommunityListing determines where an owned Asset is offered.

7\. Asset owner must have ACTIVE Membership in the target Community

   before listing.

8\. Asset has NO total\_quantity field.

9\. AssetUnits are materialized immediately at Asset creation.

10\. Asset + N AssetUnits are created in one atomic database transaction.

11\. Quantity and availability are derived from AssetUnits.

12\. A CommunityListing has no independent quantity.

13\. Public V2.1 Explore/search returns aggregate Asset summaries and

    does not expose AssetUnit IDs.

14\. One AssetUnit can participate in at most one active Transaction.

15\. Backend/database is authoritative for availability and concurrency.

16\. Client timestamps cannot establish authoritative reservation order.

17\. Physical handover is a first-class future Transaction event.

18\. Four live-photo evidence moments belong to the long-term model.

19\. Lender handover confirmation starts accountability and opens a

    limited borrower dispute window.

20\. Evidence is transaction-scoped and privacy-sensitive.

21\. Location can support community verification but should not imply

    continuous surveillance.

23\. The same creator/manager cannot have two ACTIVE Communities with

    the same name.

24\. AI inspection and tribunal are deferred until the underlying

    foundations are mature.

\`\`\`

**# 22. Core architecture statement**

\> **\*\*A user owns physical assets independently of communities. A community listing determines where an asset is offered. AssetUnits represent the actual physical quantity, and a physical unit can participate in at most one active borrowing transaction at a time.\*\***

**# 23. Pre-implementation consistency checklist**

Before any V2.1 implementation is generated:

\`\`\`text

[ ] Asset has no total\_quantity column.

[ ] Asset quantity is represented only by AssetUnits.

[ ] Asset + N AssetUnits are created atomically.

[ ] Membership.context\_metadata uses MySQL JSON.

[ ] CommunityListing checks the owner's ACTIVE membership.

[ ] Public Explore/search returns aggregate Asset summaries only.

[ ] AssetUnit IDs are not exposed in public search.

[ ] Same creator cannot create duplicate ACTIVE community names.
[ ] Community has no OPEN_FOR_ALL admission mode.
[ ] MANAGER_APPROVAL and LOCATION_VERIFIED are the only V2.1 admission modes.
[ ] verified_by is validated against an ACTIVE manager in the same Community.
[ ] Community active-name uniqueness is enforced at database level.
[ ] V2 schema uses schema.sql + ddl-auto=validate.
[ ] V2 seed uses an idempotent ApplicationRunner.

[ ] V2 starts from a fresh database with deterministic seed data.

[ ] V1 data migration is not implemented.

[ ] Backend/database owns availability and concurrency.

[ ] One physical AssetUnit cannot have two active Transactions.

\`\`\`



**## V2.1 database boundary — LOCKED**

V2.1 starts from a **\*\*fresh database\*\*** and does not migrate V1 rows.